In [2]:
import torch
import torch.nn as nn

class BinaryClassifier(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2, hidden_layers=[256, 128, 64]):
        super().__init__()

        layerlist = []
        n_in = input_size

        for h in hidden_layers:
            layerlist += [
                nn.Linear(n_in, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),
            ]
            n_in = h

        # Final output layer: 1 logit for binary classification
        layerlist.append(nn.Linear(n_in, 1))

        self.network = nn.Sequential(*layerlist)

        # Better initialization (same as before)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        # shape: (batch_size, 1) → squeeze to (batch_size,)
        logits = self.network(x).squeeze(-1)
        return logits


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 0) Load ALREADY-SCALED binary dataset
df_binary = pd.read_csv("artifacts/final_train_scaled_binary_clustered.csv")

TARGET_COL = "MP_binary_high"

# Exclude non-features
exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}

# Use only numeric feature columns
num_cols = df_binary.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

# Build X/y
X = df_binary[feature_cols].to_numpy(np.float32)
y = df_binary[TARGET_COL].to_numpy(np.float32)

# Stratification labels (cluster)
y_strat = df_binary["Structure_Cluster"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("n features:", len(feature_cols))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

# 1) Fix folds once
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr_idx, val_idx) for tr_idx, val_idx in skf.split(X, y_strat)]
print("Built folds:", len(folds))


X shape: (17625, 202)
y shape: (17625,)
n features: 202
Label counts: {np.float32(0.0): np.int64(16853), np.float32(1.0): np.int64(772)}
Built folds: 10


In [ ]:
from sklearn.metrics import accuracy_score
from pathlib import Path

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    

from scipy.special import expit  # stable sigmoid

def save_checkpoint(
    model,
    optimizer,
    epoch,
    train_loss,
    val_loss,
    y_train,
    y_val,
    val_loader,
    fold_idx,
    checkpoint_dir,
    checkpoint_tracking,
    is_final=False,
):
    # Put model in eval mode and collect logits
    model.eval()
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            logits = model(xb).cpu().numpy()
            all_logits.append(logits)

    logits_val = np.concatenate(all_logits)   # shape (N,)
    probs_val  = expit(logits_val)            # stable sigmoid
    preds_val  = (probs_val >= 0.5).astype(int)

    # y_val expected as numpy 0/1
    y_val_np = np.asarray(y_val).astype(int)

    # Classification metric: accuracy
    acc = accuracy_score(y_val_np, preds_val)

    # Filename
    if is_final:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
    else:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"

    checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename

    # Save checkpoint (weights + optimizer + losses + metric)
    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc,
        "fold_idx": fold_idx,
        "is_final": is_final,
    }
    torch.save(checkpoint_data, checkpoint_path_full)

    # Track info for CSV
    checkpoint_info = {
        "Fold": fold_idx,
        "Epoch": epoch,
        "Checkpoint_Filename": checkpoint_filename,
        "Checkpoint_Path": str(checkpoint_path_full),
        "Train_Loss": round(train_loss, 6),
        "Val_Loss": round(val_loss, 6),
        "Accuracy": round(acc, 6),
        "Is_Final": is_final,
    }
    checkpoint_tracking.append(checkpoint_info)

    checkpoint_type = "Final" if is_final else "Regular"
    print(
        f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - "
        f"Val Loss: {val_loss:.4f} | Acc: {acc:.4f}"
    )
    return True


In [5]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


def evaluate_fold(
    trial,
    fold_idx,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    hidden_layers,
    learning_rate,
    batch_size,
    dropout_rate,
    weight_decay,
    max_epochs=10**9,
    patience=30,
    min_delta=0,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_classifier",
    save_every_n_epochs=15,
):

    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu (binary classifier, BCELoss)")

    # ---- checkpoints ----
    checkpoint_tracking = []
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # ---- tensors (y must be float 0/1) ----
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # ---- model + BCELoss ----
    model = BinaryClassifier(
        input_size=X_train_scaled.shape[1],
        hidden_layers=hidden_layers,
        dropout_rate=dropout_rate,
    ).to(device)

    criterion = nn.BCELoss()  # new criterion
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

    # ---- training loop ----
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            logits = model(xb)                 # (batch,)
            probs  = torch.sigmoid(logits)     # (batch,) in [0,1]  <-- REQUIRED for BCELoss
            loss   = criterion(probs, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb)
                probs  = torch.sigmoid(logits)
                loss   = criterion(probs, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # checkpoints (your existing save_checkpoint() computes sigmoid/acc, so it still works)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader,
                fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            # final checkpoint if we didn't just save one
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                    is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # ---- load best ----
    model.load_state_dict(best_state)
    model.eval()

    # save checkpoint tracking csv
    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_classifier.csv", index=False)

    # ---- final accuracy on val set (best model) ----
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_logits.append(model(xb).cpu().numpy())
    logits_val = np.concatenate(all_logits)
    probs_val  = 1 / (1 + np.exp(-logits_val))
    preds_val  = (probs_val >= 0.5).astype(int)

    y_val_np = np.asarray(y_val).astype(int)
    acc = accuracy_score(y_val_np, preds_val)

    return best_val_loss, acc, model, train_losses, val_losses, stop_epoch

import time
import numpy as np
from pathlib import Path

trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Hidden layers (same logic)
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    val_losses = []
    accs = []

    # Run this hyperparameter combo across all folds
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]

        # Put classifier checkpoints in a classifier-specific folder
        trial_checkpoint_root = Path("checkpoints_binary_classifier") / f"trial_{trial.number:03d}"

        best_val_loss, acc, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root,
        )

        val_losses.append(best_val_loss)
        accs.append(acc)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_val_loss = float(np.mean(val_losses))
    avg_acc      = float(np.mean(accs))

    print(f"Trial {trial.number}: Avg Val Loss = {avg_val_loss:.4f} | Avg Acc = {avg_acc:.4f}")

    # Optuna minimizes this
    return avg_val_loss

In [6]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print(best_params)

/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-14 19:20:06,835] A new study created in memory with name: no-name-3cfeba63-94fb-4b47-88af-f8af1094b462


Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_000/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.4130 | Acc: 0.9217
[Fold 0] Epoch    1 | Train Loss: 0.5469 | Val Loss: 0.4130 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.1126 | Acc: 0.9586
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1042 | Acc: 0.9592
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.1018 | Acc: 0.9660
[Fold 0] Epoch   50 | Train Loss: 0.0954 | Val Loss: 0.1005 | ES 4/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.1010 | Acc: 0.9665
[Fold 0] Regular checkpoint saved at epoch 75 - Val Loss: 0.1004 | Acc: 0.9671
[Fold 0] Early stopping at epoch 76 (best Val Loss: 0.0990)
[Fold 0] Final checkpoint saved at epoch 76 - Val Loss: 0.1006 | Acc: 0.9660
Fold 1: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_0

[I 2025-12-14 19:24:47,644] Trial 0 finished with value: 0.10590743351328587 and parameters: {'dropout_rate': 0.2614526550050616, 'learning_rate': 0.00019830053261358507, 'weight_decay': 2.4896850308951163e-05, 'batch_size': 64, 'h1': 96}. Best is trial 0 with value: 0.10590743351328587.


[Fold 9] Early stopping at epoch 73 (best Val Loss: 0.0978)
[Fold 9] Final checkpoint saved at epoch 73 - Val Loss: 0.1032 | Acc: 0.9682
Trial 0 finished in 4.68 minutes
Trial 0: Avg Val Loss = 0.1059 | Avg Acc = 0.9604
Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_001/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.1307 | Acc: 0.9563
[Fold 0] Epoch    1 | Train Loss: 0.2430 | Val Loss: 0.1307 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0937 | Acc: 0.9654
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1033 | Acc: 0.9648
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.1071 | Acc: 0.9637
[Fold 0] Early stopping at epoch 45 (best Val Loss: 0.0937)
Fold 1: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_001/fold_1
[Fold 1] Regular checkpoint saved at epoch 1 - Val Loss: 0.1271 | Acc: 0.96

[I 2025-12-14 19:37:30,564] Trial 1 finished with value: 0.10249554413374946 and parameters: {'dropout_rate': 0.40868398377036486, 'learning_rate': 0.0006993877326082944, 'weight_decay': 3.222453491315206e-05, 'batch_size': 32, 'h1': 224}. Best is trial 1 with value: 0.10249554413374946.


...

[Fold 9] Early stopping at epoch 58 (best Val Loss: 0.0960)
[Fold 9] Final checkpoint saved at epoch 58 - Val Loss: 0.1060 | Acc: 0.9665
Trial 17 finished in 4.68 minutes
Trial 17: Avg Val Loss = 0.1062 | Avg Acc = 0.9609
Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_018/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.1470 | Acc: 0.9563
[Fold 0] Epoch    1 | Train Loss: 0.3791 | Val Loss: 0.1470 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.1119 | Acc: 0.9563
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1102 | Acc: 0.9620
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.1032 | Acc: 0.9665
[Fold 0] Epoch   50 | Train Loss: 0.1134 | Val Loss: 0.0953 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.0984 | Acc: 0.9682
[Fold 0] Regular checkpoint saved at epoch 75 - Val Loss: 0.1015 | Acc: 0.9677
[Fold 0] Early stopping at epoch 83

[I 2025-12-15 00:05:41,316] Trial 18 finished with value: 0.1079713560896847 and parameters: {'dropout_rate': 0.45511267002382844, 'learning_rate': 0.0003065066117325165, 'weight_decay': 0.001470474548465756, 'batch_size': 16, 'h1': 128}. Best is trial 1 with value: 0.10249554413374946.


[Fold 9] Early stopping at epoch 72 (best Val Loss: 0.1008)
[Fold 9] Final checkpoint saved at epoch 72 - Val Loss: 0.1061 | Acc: 0.9677
Trial 18 finished in 9.58 minutes
Trial 18: Avg Val Loss = 0.1080 | Avg Acc = 0.9605
Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_019/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.5597 | Acc: 0.8417
[Fold 0] Epoch    1 | Train Loss: 0.7167 | Val Loss: 0.5597 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.1279 | Acc: 0.9580
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1126 | Acc: 0.9614
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.1071 | Acc: 0.9631
[Fold 0] Epoch   50 | Train Loss: 0.1238 | Val Loss: 0.1059 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.1066 | Acc: 0.9648
[Fold 0] Regular checkpoint saved at epoch 75 - Val Loss: 0.1021 | Acc: 0.9631
[Fold 0] Regular checkpoint saved a

[I 2025-12-15 00:49:00,189] Trial 19 finished with value: 0.1055841561096843 and parameters: {'dropout_rate': 0.40230166674552553, 'learning_rate': 4.6252506544895544e-05, 'weight_decay': 0.00039430985399631574, 'batch_size': 32, 'h1': 224}. Best is trial 1 with value: 0.10249554413374946.


[Fold 9] Early stopping at epoch 151 (best Val Loss: 0.0958)
[Fold 9] Final checkpoint saved at epoch 151 - Val Loss: 0.0961 | Acc: 0.9654
Trial 19 finished in 43.31 minutes
Trial 19: Avg Val Loss = 0.1056 | Avg Acc = 0.9609
{'dropout_rate': 0.40868398377036486, 'learning_rate': 0.0006993877326082944, 'weight_decay': 3.222453491315206e-05, 'batch_size': 32, 'h1': 224}


In [9]:
print("Best Trial Number:", study.best_trial.number)
print(best_params)

Best Trial Number: 1
{'dropout_rate': 0.40868398377036486, 'learning_rate': 0.0006993877326082944, 'weight_decay': 3.222453491315206e-05, 'batch_size': 32, 'h1': 224}


In [10]:
from pathlib import Path
import torch
import pandas as pd

# BASE and artifacts_dir should already be defined (same script as before)
BASE = Path.cwd()  # transfer_learning
artifacts_dir = BASE / "artifacts"

# ---------- Directories for final best models + checkpoints ----------
best_models_dir = artifacts_dir / "binary_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_binary_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

print("Best hyperparameters from Optuna:", best_params)

# Helper to derive hidden layers from best_params (same logic as in objective)
def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay, "| batch_size:", batch_size)

final_fold_metrics = []

# ---------- Final training loop for all folds (using `folds`, X, y) ----------
# Assumes you already defined:
#   X, y, folds = [(tr_idx, val_idx), ...] earlier (same as in `objective`)
for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final training for fold {fold_idx} ====================")

    X_train_scaled = X[tr_idx]
    y_train        = y[tr_idx]
    X_val_scaled   = X[val_idx]
    y_val          = y[val_idx]

    # Train this fold with the best hyperparameters
    best_val_loss, acc, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,
        fold_idx=fold_idx,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,
        checkpoint_dir=final_ckpt_dir,   # evaluate_fold will make fold_{k}/ inside
        save_every_n_epochs=15,
    )

    # ---------- Save the final (best) model for this fold ----------
    model_path = best_models_dir / f"binary_best_fold_{fold_idx}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "fold_idx": fold_idx,
            "best_val_bce": float(best_val_loss),
            "val_accuracy": float(acc),
        },
        model_path,
    )
    print(f"[Fold {fold_idx}] Saved best classifier model to: {model_path}")
    print(f"[Fold {fold_idx}] Val BCE: {best_val_loss:.4f} | Val Acc: {acc:.4f} | Stop epoch: {stop_epoch}")

    # store metrics
    final_fold_metrics.append(
        {
            "Fold": fold_idx,
            "Best_Val_BCE": float(best_val_loss),
            "Val_Accuracy": float(acc),
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

# ---------- Save a summary CSV of all folds ----------
metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "binary_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved summary of best classifier models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'dropout_rate': 0.40868398377036486, 'learning_rate': 0.0006993877326082944, 'weight_decay': 3.222453491315206e-05, 'batch_size': 32, 'h1': 224}
Using hidden_layers: [224, 112, 56]
dropout: 0.40868398377036486 | lr: 0.0006993877326082944 | wd: 3.222453491315206e-05 | batch_size: 32

==================== Final training for fold 0 ====================
Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/checkpoints_binary_best/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.1264 | Acc: 0.9552
[Fold 0] Epoch    1 | Train Loss: 0.2700 | Val Loss: 0.1264 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.1018 | Acc: 0.9614
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1082 | Acc: 0.9637
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.1089 | Acc: 0.9648
[Fold 0] Epoch   50 | Train Loss: 0.0688 | Val Loss: 0.1

In [11]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit

confusion_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Confusion matrix for fold {fold_idx} =====")

    # Load validation data
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # Load best model
    model_path = best_models_dir / f"binary_best_fold_{fold_idx}.pt"
    checkpoint = torch.load(model_path, map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=checkpoint["hidden_layers"],
        dropout_rate=checkpoint["dropout_rate"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Predict
    with torch.no_grad():
        logits = model(X_val_tensor).cpu().numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    confusion_matrices.append(cm)

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")




===== Confusion matrix for fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1671   15]
 [  47   30]]
TN=1671, FP=15, FN=47, TP=30

===== Confusion matrix for fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1682   11]
 [  53   17]]
TN=1682, FP=11, FN=53, TP=17

===== Confusion matrix for fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1657   28]
 [  40   38]]
TN=1657, FP=28, FN=40, TP=38

===== Confusion matrix for fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1683   18]
 [  43   19]]
TN=1683, FP=18, FN=43, TP=19

===== Confusion matrix for fold 4 =====
Confusion matrix (rows=true, cols=pred):
[[1662   12]
 [  60   29]]
TN=1662, FP=12, FN=60, TP=29

===== Confusion matrix for fold 5 =====
Confusion matrix (rows=true, cols=pred):
[[1675   12]
 [  58   17]]
TN=1675, FP=12, FN=58, TP=17

===== Confusion matrix for fold 6 =====
Confusion matrix (rows=true, cols=pred):
[[1633   28]
 [  60   41]]
TN=1633, FP=28, FN=60, TP=41

===== Confusion matrix for fold 7

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_18417/2467034275.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_loca

In [12]:
mean_cm = np.mean(confusion_matrices, axis=0)

mean_cm_df = pd.DataFrame(
    mean_cm,
    index=["True Low&Int", "True High"],
    columns=["Pred Low&Int", "Pred High"],
)

print("\n===== Mean confusion matrix across folds =====")
print(mean_cm_df)

# Save to CSV
mean_cm_path = best_models_dir / "binary_mean_confusion_matrix.csv"
mean_cm_df.to_csv(mean_cm_path)
print(f"\n✅ Saved mean confusion matrix to: {mean_cm_path}")



===== Mean confusion matrix across folds =====
              Pred Low&Int  Pred High
True Low&Int        1669.2       16.1
True High             51.8       25.4

✅ Saved mean confusion matrix to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/artifacts/binary_best_models/binary_mean_confusion_matrix.csv


In [13]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit  # stable sigmoid

def metrics_from_confusion(cm):
    """
    cm = [[TN, FP],
          [FN, TP]]
    """
    TN, FP, FN, TP = cm.ravel()

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": specificity,
    }

all_fold_metrics = []
all_conf_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\nEvaluating fold {fold_idx}")

    # Load best model for this fold
    ckpt = torch.load(best_models_dir / f"binary_best_fold_{fold_idx}.pt", map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=ckpt["hidden_layers"],
        dropout_rate=ckpt["dropout_rate"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Validation data
    X_val = torch.tensor(X[val_idx], dtype=torch.float32)
    y_val = y[val_idx].astype(int)

    # Predict
    with torch.no_grad():
        logits = model(X_val).numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_val, preds)
    all_conf_matrices.append(cm)

    # Metrics
    metrics = metrics_from_confusion(cm)
    metrics["Fold"] = fold_idx
    all_fold_metrics.append(metrics)

    print("Confusion matrix:\n", cm)
    print(metrics)



Evaluating fold 0
Confusion matrix:
 [[1671   15]
 [  47   30]]
{'Accuracy': np.float64(0.9648326715825298), 'Precision': np.float64(0.6666666666666666), 'Recall': np.float64(0.38961038961038963), 'F1': np.float64(0.4918032786885245), 'Specificity': np.float64(0.9911032028469751), 'Fold': 0}

Evaluating fold 1
Confusion matrix:
 [[1682   11]
 [  53   17]]
{'Accuracy': np.float64(0.9636982416335791), 'Precision': np.float64(0.6071428571428571), 'Recall': np.float64(0.24285714285714285), 'F1': np.float64(0.3469387755102041), 'Specificity': np.float64(0.993502658003544), 'Fold': 1}

Evaluating fold 2
Confusion matrix:
 [[1657   28]
 [  40   38]]
{'Accuracy': np.float64(0.9614293817356778), 'Precision': np.float64(0.5757575757575758), 'Recall': np.float64(0.48717948717948717), 'F1': np.float64(0.5277777777777778), 'Specificity': np.float64(0.9833827893175074), 'Fold': 2}

Evaluating fold 3
Confusion matrix:
 [[1683   18]
 [  43   19]]
{'Accuracy': np.float64(0.9653998865570051), 'Precisio

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_18417/1914565896.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_models_dir / f"binary

In [14]:
metrics_df = pd.DataFrame(all_fold_metrics)

mean_metrics = metrics_df.mean(numeric_only=True)
std_metrics  = metrics_df.std(numeric_only=True)

summary_df = pd.DataFrame({
    "Mean": mean_metrics,
    "Std": std_metrics
})

print("\n=== Cross-validated performance ===")
print(summary_df)

summary_df.to_csv(
    best_models_dir / "binary_classifier_confusion_metrics_summary.csv"
)



=== Cross-validated performance ===
                 Mean       Std
Accuracy     0.961475  0.004648
Precision    0.621636  0.080951
Recall       0.324801  0.085421
F1           0.417997  0.068191
Specificity  0.990437  0.004555
Fold         4.500000  3.027650
